In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PartitionDemo") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 05:48:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/13 05:48:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/13 05:48:59 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [2]:
data = [(i, i*10) for i in range(20)]

df = spark.createDataFrame(data, ["id", "value"])

In [3]:
print("Partitions:", df.rdd.getNumPartitions())

Partitions: 2


In [4]:
df_repartitioned = df.repartition(20)
print("Partitions:", df_repartitioned.rdd.getNumPartitions())

[Stage 0:>                                                          (0 + 2) / 2]

Partitions: 20


[Stage 0:=============================>                             (1 + 1) / 2]

In [5]:
print(df_repartitioned.rdd.glom().map(len).collect())

[Stage 2:=============================================>           (16 + 2) / 20]

[0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 0]


In [5]:
print(df_repartitioned.rdd.glom().map(len).collect())

[Stage 2:=============================================>           (16 + 2) / 20]

[0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 0]


In [52]:
df = spark.range(100000)
print("Before:", df.rdd.getNumPartitions())
df = df.repartition(20)
print("After:", df.rdd.getNumPartitions())

Before: 2
After: 20


In [7]:
df.take(20) 

[Row(id=0, value=0),
 Row(id=1, value=10),
 Row(id=2, value=20),
 Row(id=3, value=30),
 Row(id=4, value=40),
 Row(id=5, value=50),
 Row(id=6, value=60),
 Row(id=7, value=70),
 Row(id=8, value=80),
 Row(id=9, value=90),
 Row(id=10, value=100),
 Row(id=11, value=110),
 Row(id=12, value=120),
 Row(id=13, value=130),
 Row(id=14, value=140),
 Row(id=15, value=150),
 Row(id=16, value=160),
 Row(id=17, value=170),
 Row(id=18, value=180),
 Row(id=19, value=190)]

In [12]:
# optimization and caching

In [16]:
optimized_df = df.filter(col("value") > 500) \
                 .filter(col("id") < 5000000) \
                 .select("id", "value")

In [17]:
optimized_df.explain()

== Physical Plan ==
*(1) Filter ((isnotnull(value#1L) AND isnotnull(id#0L)) AND ((value#1L > 500) AND (id#0L < 5000000)))
+- *(1) Scan ExistingRDD[id#0L,value#1L]




In [20]:
start_time = time.time()
count_uncached = optimized_df.count()
end_time = time.time()
print(f"1. Optimized Execution | Count: {count_uncached} | Time: {round(end_time - start_time, 4)} seconds")

1. Optimized Execution | Count: 0 | Time: 0.3687 seconds


In [21]:
start_time = time.time()
count_cached = optimized_df.count()
end_time = time.time()
print(f"2. Cached Execution | Count: {count_cached} | Time: {round(end_time - start_time, 4)} seconds")

2. Cached Execution | Count: 0 | Time: 0.3955 seconds


In [22]:
# udfs

In [48]:
data = [
    ("Alice", 25),
    ("Bob", 17),
    ("Charlie", 20)
]

df = spark.createDataFrame(data, ["name", "age"])
df.show()

+-------+---+
|   name|age|
+-------+---+
|  Alice| 25|
|    Bob| 17|
|Charlie| 20|
+-------+---+



In [49]:
def can_drive(age):
    if age >= 18:
        return "Can Drive"
    else:
        return "Cannot Drive"

In [50]:
drive_udf = udf(can_drive, StringType())

In [51]:
df_new = df.withColumn("Driving_Status", drive_udf(df["Age"]))
df_new.show()

+-------+---+--------------+
|   name|age|Driving_Status|
+-------+---+--------------+
|  Alice| 25|     Can Drive|
|    Bob| 17|  Cannot Drive|
|Charlie| 20|     Can Drive|
+-------+---+--------------+



In [53]:
spark.stop()